# Job Task 2: Build the Vector Search source table

Writes `financial_docs_for_search` as a managed Delta table with Change Data Feed enabled.
This table is the source for the Vector Search Delta Sync index used by the Agent Bricks
Knowledge Assistant in Session 2.

Why a separate table rather than indexing `financial_docs_silver` directly?
- `financial_docs_silver` is a Streaming Table; Vector Search Delta Sync requires CDF,
  which is not compatible with SDP streaming tables.
- This job task reads from the completed silver table and writes a plain managed Delta table
  with `delta.enableChangeDataFeed = true`, making it a valid Delta Sync source.

In [0]:
dbutils.widgets.text("catalog",       "platform-workshop", "Catalog")
dbutils.widgets.text("shared_schema", "00_shared",         "Shared Schema")

catalog       = dbutils.widgets.get("catalog")
shared_schema = dbutils.widgets.get("shared_schema")

c = f"`{catalog}`"
print(f"Building search table in: {catalog}.{shared_schema}")

In [ ]:
# Write financial_docs_for_search as a plain managed Delta table.
# Overwrite mode ensures the table stays in sync with silver on every pipeline run.
# silver already stores plain_text (extracted directly from the ai_parse_document VARIANT),
# so no further transformation is needed here.
(
    spark.read.table(f"{c}.{shared_schema}.financial_docs_silver")
    .select(
        "source_path",   # Primary key for the Vector Search index
        "plain_text",    # Clean prose text — ready to embed
        "company",       # Metadata filter dimension
        "fiscal_period", # Metadata filter dimension
        "document_type", # Metadata filter dimension
    )
    .filter("plain_text IS NOT NULL")
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{c}.{shared_schema}.financial_docs_for_search")
)

print(f"✓ financial_docs_for_search written")

In [0]:
# Enable Change Data Feed — required for Vector Search Delta Sync.
# ALTER TABLE is idempotent; safe to run on every job execution.
spark.sql(f"""
    ALTER TABLE {c}.{shared_schema}.financial_docs_for_search
    SET TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'quality'                    = 'gold'
    )
""")

row_count = spark.sql(
    f"SELECT COUNT(*) AS cnt FROM {c}.{shared_schema}.financial_docs_for_search"
).collect()[0]['cnt']

print(f"✓ CDF enabled on financial_docs_for_search")
print(f"  Rows available for indexing: {row_count:,}")